# Step 3: SmoothQuant — 等价缩放迁移

**目标**：实现 SmoothQuant（Xiao 2022）的等价缩放变换 $Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$，把激活的 emergent outlier"迁移"到权重里——让激活变平滑、可全 INT8 量化。理解它为何是 LLM.int8()（s2）的"零代价升级"。

**对应 OUTLINE 课时**：1.3 SmoothQuant（~55 分钟）。

> **事实纠正（重要）**：OUTLINE 旧文写"llmcompressor 默认 smoothing_strength=0.8（非论文 0.5）"——**这是错的**。实测 llmcompressor 0.12.0：`SmoothQuantModifier()` 类默认 `smoothing_strength=0.5`（=论文 α）；0.8 是课程**刻意取**、需显式传。本节讲原理时用正确表述。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理

s2 的 LLM.int8() 用"分流"处理 outlier，代价是运行时双路径。SmoothQuant 的洞察：**不必分流——把 outlier 从激活搬进权重**，用一次等价数学变换。

### 等价变换

对线性层 $Y = X W$（$X \in \mathbb{R}^{m \times k}$，$W \in \mathbb{R}^{k \times n}$），SmoothQuant 引入对角缩放向量 $s \in \mathbb{R}^{k}$（每 channel 一个）：
$$Y = X W = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W) = \tilde{X} \tilde{W}$$
矩阵乘结果**完全不变**（纯代数恒等式），但：
- $\tilde{X} = X / s$：除掉 outlier channel 的 $s_j$（大值），激活被"压平"。
- $\tilde{W} = s \cdot W$：乘上 $s_j$，权重该 channel 被放大——但权重本来分布平稳，放大一点通常不破坏 INT8。

### 缩放向量公式

$s$ 要平衡两边：让 $\tilde{X}$ 和 $\tilde{W}$ 的"量化难度"相当。SmoothQuant 取：
$$s_j = \frac{\max(|X_{:,j}|)^{\alpha}}{\max(|W_{:,j}|)^{1-\alpha}}, \quad \alpha \in [0,1]$$
- $\alpha$ 控制迁多少给权重。$\alpha=0$：完全不迁（等于原状）；$\alpha=1$：全迁给权重。
- **论文默认 $\alpha=0.5$**（几何均值，两边各迁一半）。课程常取 $\alpha=0.8$（多迁给权重——因为权重更稳，多扛一点通常无损）。
- 分子用激活 channel $j$ 的 max abs，分母用权重 channel $j$（=W 第 j 行）的 max abs。

### α 越大越好吗？——不一定，α 是两个目标的权衡（L3 要亲眼验证）

$\alpha$ 调的是两个互相牵制的目标：
1. **压平激活 outlier**：α 越大，$s_j$ 越受激活主导、越大，激活除得越多、outlier 通道被压得越平（这正是 SmoothQuant 想要的）；
2. **整体 INT8 相对误差**：α 过大会把 $\tilde{W}$ 某些 channel 放大到接近/超出 INT8 表示范围，反而拉高整体量化误差。

在「单层、整体相对误差」这个指标上（合成数据 + 单层 v_proj 都属此类），**α=0.5（几何均值、两边均衡）通常最优或并列最优**——你会在 L2a 的 `report_smooth()` 里看到这点：弱/强 outlier 两种场景下 α=0.5 的整体 INT8 误差都 ≤ α=0.8。

那 α=0.8 的价值在哪？在于**更激进地压平强 outlier 通道**——这些通道幅值极大、对下游 perplexity 影响远超其数量占比。牺牲一点整体误差来确保最关键的几个通道不被量化打爆，在强 outlier 的大模型上（如 s1 实测 7B 的 gate/up_proj，max/mean≈58×）通常是划算的，故论文推荐大模型取 α≈0.8。

> **判断要点（重要，破除"α 越大越好"的误解）**：
> - α 不是"越大越优"，而是"匹配 outlier 强度与目标"。
> - 看整体单层 INT8 误差 → α=0.5 常胜（均衡）。
> - 看下游 ppl / 强 outlier 保护 → 大模型取 α≈0.8。
> - **L3 你会在真 0.5B v_proj 上亲手看到反例**：v_proj 的激活 outlier 较弱，α=0.8 过度迁移，整体 INT8 误差反而 > α=0.5。这不是 SmoothQuant 失效，是 α 选错了。
> - 工业实践：α 按每层/每投影独立搜索，而非全局固定一个值。

这是 SmoothQuant 相对 LLM.int8() 的关键优势：迁移后**整条路径都能纯 INT8**，无双路径开销——前提是 α 选对。

## 本步填空

1. **`compute_smooth_scale(x_abs_max, w_abs_max, alpha)`** —— 实现 $s_j = \frac{x_{\max,j}^{\alpha}}{w_{\max,j}^{1-\alpha}}$ 公式。
2. **`apply_smooth_transform(x, w, s)`** —— 返回等价变换后的 $(\tilde{X}, \tilde{W})$，并验证 $\tilde{X}\tilde{W} = XW$。
3. （判断，含在测试里）理解 $\alpha$ 如何影响"激活难度下降比"。

> **关于 α 的两个默认值（重要，避免困惑"到底用哪个 α"）**
>
> 本 notebook 里 α 出现在**两个函数**、**默认值不同**，这不是疏忽，是分工：
>
> | 函数 | 默认 α | 角色 | 为什么是这个默认 |
> |------|--------|------|------------------|
> | `compute_smooth_scale(alpha=0.5)` | **0.5** | **纯公式层**：给定 x_max/w_max 算 s，不关心数据从哪来。0.5 = 论文默认、几何均值（两边各迁一半），是"公式基准值"。 | 对齐 SmoothQuant 论文 §3.3 的 $s_j$ 定义式，公式层面的中性默认。 |
> | `fit_smooth_scale(alpha=0.8)` | **0.8** | **数据拟合层**：从真实激活+权重 fit 出 s 时用。0.8 = 课程对大模型（强 outlier）的**取值选择**，多迁给权重以保护激活关键通道。 | 匹配 s1 实测 7B gate/up_proj 的 58× 强 outlier；工业上大模型常取 0.8-0.85。 |
>
> **一句话**：`compute_smooth_scale` 的 0.5 是"公式的中立基准"（你要实现的数学语义），`fit_smooth_scale` 的 0.8 是"应用到真实大模型时的工程取值"。L2/L3 里你会显式对比 0.5 vs 0.8，亲手看到为什么强 outlier 下取 0.8、弱 outlier 下取 0.5——这正是判断要点。两个默认值不同恰恰是教学设计，让你留意 α 是可调超参，而非写死。

In [ ]:
def compute_smooth_scale(x_abs_max, w_abs_max, alpha=0.5):
    """SmoothQuant 缩放向量 s_j = x_max[j]^alpha / w_max[j]^(1-alpha)。

    参数
    ----
    x_abs_max : torch.Tensor [k]，每个激活 channel 的 max|X[:,j]|。
    w_abs_max : torch.Tensor [k]，每个权重 channel（W 的行 j）的 max|W[j,:]|。
    alpha : float in [0,1]，迁移强度（论文默认 0.5，课程常取 0.8）。

    返回
    ----
    torch.Tensor [k]，每个 channel 的平滑 scale s_j，float32。
      数值要保证非负、无 inf/nan（防 0 除：w_max 用 clamp_min(eps)）。

    提示
    ----
      - 直接套公式：s = x_max ** alpha / (w_max ** (1 - alpha))。
      - 防 0 除：w_abs_max 先 .clamp_min(1e-8)（或加 eps）再取幂。
      - torch 的 ** 对 tensor 是逐元素幂，OK。
      - 思考：alpha=0 时 s = 1/w_max（全迁激活侧的反向？不——alpha=0 时分子 x^0=1，
        s=1/w_max，激活除以 1/w_max=乘 w_max，反而放大激活。说明 alpha=0 不合理，
        实践 alpha∈[0.5,0.85]。这个边界思考就是"判断型"训练。）
    """
    # TODO: 实现 SmoothQuant 的 s 公式，返回 [k] 张量。
    raise NotImplementedError


def apply_smooth_transform(x, w, s):
    """对 X,W 应用等价缩放变换，返回 (X_tilde, W_tilde)。

    参数
    ----
    x : torch.Tensor [m, k] 激活。
    w : torch.Tensor [k, n] 权重。
    s : torch.Tensor [k] 平滑向量（来自 compute_smooth_scale）。

    返回
    ----
    (x_tilde, w_tilde) 元组：
      - x_tilde : [m, k] = X / s[j]（s 沿 channel 维广播：s 形状 [k] → [1,k]）。
      - w_tilde : [k, n] = s[j] * W（s 形状 [k] → [k,1]）。

    数学保证：x_tilde @ w_tilde == x @ w（等价）。测试会检查这点。

    提示
    ----
      - X 除 s：s.unsqueeze(0) 广播成 [1,k]，x / s.unsqueeze(0)。
      - W 乘 s：s.unsqueeze(1) 广播成 [k,1]，w * s.unsqueeze(1)。
      - 注意 dtype 统一 float32。
    """
    # TODO: 返回 (X/s, s*W)。
    raise NotImplementedError


# 脚手架（提供）：从一批激活 + 权重算出 s（取 max|·| 喂 compute_smooth_scale）。
def fit_smooth_scale(x, w, alpha=0.8):
    """从数据算 s：x_abs_max 每 channel 的 max，w_abs_max 每 channel 的 max。"""
    x_abs_max = x.float().abs().amax(dim=0)         # [k]
    w_abs_max = w.float().abs().amax(dim=1)         # [k]
    return compute_smooth_scale(x_abs_max, w_abs_max, alpha=alpha)

# 脚手架（提供）：smooth 后用 vector-wise INT8 量化 matmul（复用 s2 思路，本 notebook 内联精简版）
def _int8_matmul(x, w):
    def vwq(t):
        rm = t.abs().amax(1, keepdim=True).clamp_min(1e-12); sc = 127.0/rm.squeeze(1)
        return (torch.round(t*sc.unsqueeze(1)).clamp(-127,127).to(torch.int8), sc)
    xq,cx = vwq(x); wq,cw = vwq(w.t())
    return (xq.to(torch.int32)@wq.to(torch.int32).t()).float()/(cx.unsqueeze(1)*cw.unsqueeze(0))

In [ ]:
%%ipytest -qq

def test_compute_smooth_scale_value():
    x_max = torch.tensor([8.0, 2.0])   # k=2
    w_max = torch.tensor([2.0, 8.0])
    s = compute_smooth_scale(x_max, w_max, alpha=0.5)
    # s0 = sqrt(8/2)=2 ; s1 = sqrt(2/8)=0.5
    assert torch.allclose(s, torch.tensor([2.0, 0.5]), atol=1e-4)

def test_compute_smooth_scale_alpha_skew():
    x_max = torch.tensor([8.0]); w_max = torch.tensor([2.0])
    s_half = compute_smooth_scale(x_max, w_max, alpha=0.5)
    s_high = compute_smooth_scale(x_max, w_max, alpha=0.8)
    # alpha=0.8: s = 8^0.8 / 2^0.2 ≈ 5.278 / 1.149 ≈ 4.59 > alpha=0.5 的 2.0
    assert s_high.item() > s_half.item(), "alpha 越大 s 越大（更多迁给权重）"

def test_compute_smooth_scale_no_nan_on_zero_w():
    x_max = torch.tensor([8.0, 2.0]); w_max = torch.tensor([0.0, 8.0])  # 有 0
    s = compute_smooth_scale(x_max, w_max, alpha=0.5)
    assert not torch.isnan(s).any() and not torch.isinf(s).any()

def test_apply_smooth_transform_equivalence():
    torch.manual_seed(0)
    x = torch.randn(4, 8); w = torch.randn(8, 6)
    s = torch.tensor([2.0]*8)
    xt, wt = apply_smooth_transform(x, w, s)
    y_orig = x @ w; y_smooth = xt @ wt
    assert torch.allclose(y_orig, y_smooth, atol=1e-5), "等价变换必须保证 XW == X_tilde W_tilde"

def test_apply_smooth_transform_shapes():
    x = torch.randn(4, 8); w = torch.randn(8, 6); s = torch.rand(8)+0.5
    xt, wt = apply_smooth_transform(x, w, s)
    assert xt.shape == x.shape and wt.shape == w.shape

def test_smooth_reduces_activation_difficulty():
    """判断型：smooth 后激活的 max/mean 应下降（outlier 被吸收），权重略升但可控。"""
    torch.manual_seed(1)
    x = torch.randn(16, 32); w = torch.randn(32, 16)
    x[:, 5] *= 30.0   # 注入 outlier channel
    s = fit_smooth_scale(x, w, alpha=0.8)
    xt, wt = apply_smooth_transform(x, w, s)
    diff_before = x.abs().amax(dim=0).mean()
    diff_after = xt.abs().amax(dim=0).mean()
    assert diff_after.item() < diff_before.item(), "smooth后激活难度应下降"


## L2：tiny 验证（CPU）—— forward 等价 err~1e-7 + 激活难度下降

用合成 + tiny Qwen2 验证：① SmoothQuant 变换前后 `XW` 数值等价（atol 极小）；② smooth 后再走 INT8，精度远好于不平滑直接 INT8。

In [ ]:
# 等价性验证：变换前后 XW 几乎完全相等
torch.manual_seed(3)
x = torch.randn(8, 16); w = torch.randn(16, 12)
x[:, 2] *= 30.0; x[:, 9] *= 20.0
s = fit_smooth_scale(x, w, alpha=0.8)
xt, wt = apply_smooth_transform(x, w, s)
y_orig = x @ w; y_smooth = xt @ wt
equiv_err = (y_orig - y_smooth).abs().max().item()
print(f"SmoothQuant 等价误差 max|XW - XtWt| = {equiv_err:.2e}（应 ~1e-7，纯浮点误差）")
assert equiv_err < 1e-5, "等价变换误差应极小"

# smooth 后 INT8 精度 vs 不 smooth 直接 INT8
y_ref = x @ w
y_int8_raw = _int8_matmul(x, w)
y_int8_smooth = _int8_matmul(xt, wt)
err_raw = ((y_int8_raw - y_ref).abs().mean()/y_ref.abs().mean()).item()
err_smooth = ((y_int8_smooth - y_ref).abs().mean()/y_ref.abs().mean()).item()
print(f"不 smooth 直接 INT8 相对误差: {err_raw:.4f}")
print(f"smooth 后 INT8 相对误差: {err_smooth:.4f}（应明显更小）")
assert err_smooth < err_raw
print("L2a PASS：SmoothQuant 等价 + smooth 后 INT8 精度提升")

# tiny Qwen2：取一层 v_proj 验证 forward 等价
from transformers import Qwen2Config, Qwen2ForCausalLM
def make_tiny(vocab=320, hidden=128, inter=256):
    cfg = Qwen2Config(num_hidden_layers=1, hidden_size=hidden, intermediate_size=inter,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=vocab, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()
tiny = make_tiny()
vproj = tiny.model.layers[0].self_attn.v_proj
X = torch.randn(4, tiny.config.hidden_size)
W = vproj.weight.data.float().t()
s = fit_smooth_scale(X, W, alpha=0.8)
Xt, Wt = apply_smooth_transform(X, W, s)
err = ((X@W - Xt@Wt).abs().max()).item()
print(f"\ntiny v_proj 等价误差: {err:.2e}")
assert err < 1e-5
print("L2b PASS：tiny Qwen2 v_proj 的 SmoothQuant 等价变换成立")

## L3：H200 执行（真 0.5B v_proj，α=0.5 vs 0.8）

GPU 守卫。在真 0.5B 的一层 v_proj 上：取一段文本 forward 抓激活，对比 α=0.5（论文默认）与 α=0.8（课程取）下，smooth 后 INT8 的精度与激活难度下降比。

> **你会看到一个"反例"——这正是本步的重点**：v_proj 的激活 outlier 较弱（远没有 s1 真 7B 的 gate/up_proj 那种 58× 的强 outlier）。按上面原理的判断要点，弱 outlier 下 **α=0.5 应优于 α=0.8**：α=0.8 会过度迁移，激活难度不降反升、smooth-INT8 相对误差更大。亲手跑出的数字会印证这一点——这不是 SmoothQuant 失效，而是说明 **α 必须匹配 outlier 强度，不能无脑取大**。工业实践中 α 是每层/每投影独立搜索的超参，而非全局固定。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(TINY_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, dtype=torch.float16,
                                                 device_map="auto").eval()
    ids = tok("Machine learning models can be", return_tensors="pt").input_ids.to(model.device)
    # 抓 v_proj 输入激活
    cap = {}
    def hk(mod, inp, out):
        if not cap: cap.update(x=inp[0].detach().float().cpu())
    h = model.model.layers[0].self_attn.v_proj.register_forward_hook(hk)
    with torch.no_grad(): model(ids)
    h.remove()
    X = cap["x"].reshape(-1, cap["x"].shape[-1])
    W = model.model.layers[0].self_attn.v_proj.weight.data.float().cpu().t()
    y_ref = X @ W

    for alpha in [0.5, 0.8]:
        s = fit_smooth_scale(X, W, alpha=alpha)
        Xt, Wt = apply_smooth_transform(X, W, s)
        equiv = (X@W - Xt@Wt).abs().max().item()
        diff_before = X.abs().amax(0).mean().item()
        diff_after = Xt.abs().amax(0).mean().item()
        y_int8 = _int8_matmul(Xt, Wt)
        err = ((y_int8 - y_ref).abs().mean()/y_ref.abs().mean()).item()
        print(f"α={alpha}: 等价误差={equiv:.1e}, 激活难度 {diff_before:.2f}→{diff_after:.2f} "
              f"(降 {(1-diff_after/diff_before)*100:.0f}%), smooth-INT8 相对误差={err:.4f}")
    del model; torch.cuda.empty_cache()
else:
    print("跳过 L3：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查

打印 α=0.5 vs 0.8 的对比结论，写 smooth_summary.json。

In [ ]:
def report_smooth():
    """对比 α=0.5 vs 0.8 在「弱 outlier」与「强 outlier」两种激活下的表现。

    关键判断：α 不是越大越好，它衡量的是两个目标的权衡——
      (1) 压平激活 outlier（α 越大压得越狠，outlier 通道越平滑）；
      (2) 整体 smooth-INT8 相对误差（α 过大会把权重某些 channel 放大到超出 INT8 表示范围，反伤精度）。
    在合成 + 单层 v_proj 这种「整体相对误差」指标上，α=0.5（几何均值、两边均衡）通常最优或并列最优；
    而 α=0.8 的价值在于更激进地保护强 outlier 通道（对下游 ppl 更关键），代价是整体误差略升。
    本报告两组数据并列展示，让打印数字与结论自洽。
    """
    torch.manual_seed(11)
    summary = {}
    for scenario, outlier_scale in [("weak_outlier", 3.0), ("strong_outlier", 35.0)]:
        # 同一组基线权重，仅激活 outlier 强度不同（模拟不同层/不同投影的真实差异）
        w = torch.randn(64, 32)
        x = torch.randn(32, 64)
        x[:, 7] *= outlier_scale      # 注入一个 outlier channel
        x[:, 21] *= outlier_scale * 0.8
        for alpha in [0.5, 0.8]:
            s = fit_smooth_scale(x, w, alpha=alpha)
            Xt, Wt = apply_smooth_transform(x, w, s)
            y_ref = x@w; y_i = _int8_matmul(Xt, Wt)
            summary[f"{scenario}/alpha_{alpha}"] = {
                "equiv_max_err": float((x@w - Xt@Wt).abs().max()),
                "act_difficulty_before": float(x.abs().amax(0).mean()),
                "act_difficulty_after": float(Xt.abs().amax(0).mean()),
                "smooth_int8_relerr": float(((y_i-y_ref).abs().mean()/y_ref.abs().mean())),
            }
    (OUT_ROOT/"smooth_summary.json").write_text(json.dumps(summary, indent=2))

    print("== SmoothQuant α 对比（弱 vs 强 outlier）==")
    for scenario in ["weak_outlier", "strong_outlier"]:
        print(f"\n[{scenario}]")
        for alpha in [0.5, 0.8]:
            v = summary[f"{scenario}/alpha_{alpha}"]
            drop = (1 - v["act_difficulty_after"]/v["act_difficulty_before"])*100
            print(f"  α={alpha}: 等价误差={v['equiv_max_err']:.1e}, 激活难度降 {drop:+.0f}%, "
                  f"smooth-INT8 相对误差={v['smooth_int8_relerr']:.4f}")
        e5  = summary[f"{scenario}/alpha_0.5"]["smooth_int8_relerr"]
        e8  = summary[f"{scenario}/alpha_0.8"]["smooth_int8_relerr"]
        d5  = (1 - summary[f"{scenario}/alpha_0.5"]["act_difficulty_after"]/summary[f"{scenario}/alpha_0.5"]["act_difficulty_before"])*100
        d8  = (1 - summary[f"{scenario}/alpha_0.8"]["act_difficulty_after"]/summary[f"{scenario}/alpha_0.8"]["act_difficulty_before"])*100
        print(f"  → 整体 INT8 误差：α=0.5 {'≤' if e5<=e8 else '>'} α=0.8；激活压平幅度：α=0.8 ({d8:+.0f}%) 比 α=0.5 ({d5:+.0f}%) 更狠")

    print("\n结论（数据自洽）：")
    print("  • 在「单层整体 INT8 相对误差」指标上，α=0.5（几何均值、两边均衡）通常最优或并列最优——")
    print("    这与 L3 真 0.5B v_proj 看到的一致（弱 outlier 下 α=0.5 胜 α=0.8）。")
    print("  • α=0.8 的价值不在「整体误差更低」，而在「更激进压平强 outlier 通道」——")
    print("    这些通道对下游 perplexity 影响最大，故论文在强 outlier 大模型上推荐 α≈0.8。")
    print("  • 判断要点：α 是数据相关的调参（匹配 outlier 强度），不是越大越好；")
    print("    工业实践按层/投影独立搜 α，而非全局固定。")

report_smooth()